In [ ]:
# ============================================================================
# Persian ASR — Colloquial Transcript Normalization Study
# Does normalizing transcripts to colloquial Persian improve Whisper ASR?
#
# Four-way comparison:
#   Condition 1: Baseline            (no fine-tuning, original transcripts)
#   Condition 2: Colloquial refs     (no fine-tuning, both sides normalized)
#   Condition 3: Fine-tuned original (trained on original transcripts)
#   Condition 4: Fine-tuned colloquial (trained on colloquial-normalized transcripts)
#
# Evaluation rule (applied in ALL conditions):
#   normalize_to_colloquial() is applied to BOTH hypothesis and reference
#   before any WER/CER calculation. This removes orthographic noise
#   (spacing, ZWNJ, Arabic vs. Persian characters) so we measure acoustic
#   accuracy, not typographic convention.
#
# Based on/adapted from:
#   Hameed et al. (2025) — DOLMA-NLP/asr (Interspeech 2025)
#   https://github.com/DOLMA-NLP/asr

In [ ]:
# STAGE 1: SETUP AND ENVIRONMENT CONFIGURATION
# ============================================================================

!pip install transformers datasets evaluate accelerate jiwer soundfile librosa \
             huggingface_hub sentencepiece --quiet
!pip install torch torchaudio --quiet
!pip install datasets[audio] --quiet

from huggingface_hub import notebook_login
notebook_login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.4 MB/s eta 0:00:00


In [ ]:
# STAGE 2: LOAD DATASET — 3,000 SAMPLES
# ============================================================================

from datasets import load_dataset, Audio, Dataset

ds = load_dataset(
    "kiarashQ/farsi-asr-unified-cleaned",
    split="train",
    streaming=True
)
ds = ds.cast_column("audio", Audio(sampling_rate=16000))
print("Dataset loaded in streaming mode.")

samples = []
for i, sample in enumerate(ds):
    samples.append(sample)
    if i >= 2999:           # 3,000 samples (reduced from 10,000 for Colab stability)
        break
    if (i + 1) % 500 == 0:
        print(f"  Collected {i+1}/3000 samples...")

print(f"Collected {len(samples)} samples")
print("Columns:", list(samples[0].keys()))
print("Example transcript:", samples[0]["text"])

dataset_full = Dataset.from_list(samples)
dataset_full = dataset_full.cast_column("audio", Audio(sampling_rate=16000))
print(dataset_full)

README.md:   0%|          | 0.00/7.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/321 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/321 [00:00<?, ?it/s]

Dataset loaded in streaming mode.
  Collected 500/3000 samples...
  Collected 1000/3000 samples...
  Collected 1500/3000 samples...
  Collected 2000/3000 samples...
  Collected 2500/3000 samples...
Collected 3000 samples
Columns: ['path', 'text', 'audio', 'sampling_rate']
Example transcript: شاید کسی که از او کاری خواسته ایم نتواند
Dataset({
    features: ['path', 'text', 'audio', 'sampling_rate'],
    num_rows: 3000
})


In [ ]:
# STAGE 3: NORMALIZER — formal/noisy → colloquial
#
# Direction: FROM formal written Persian (می‌کند, نمی‌دانم) or noisy
# colloquial (می کنه, نمیدونم) TO normalized colloquial (می‌کنه, نمی‌دونم).
#
# The normalizer is organized into SEVEN literature-grounded categories,
# applied in a fixed order (order matters — later rules depend on earlier
# ones having run). Each category is documented with its source and risk
# level. The normalizer is IDEMPOTENT: running it twice on already-
# normalized text produces the same result, which matters because it is
# applied to both references AND model hypotheses (which may already be
# partially colloquial) before every WER/CER calculation.
#
# Category overview:
#   7. Arabic → Persian character normalization      [Rasooli 2013]
#   2. می / نمی ZWNJ spacing canonicalization          [Rasooli 2013, ParsisNLP]
#   1. /ān/ → [un] vowel raising                      [Miller 2011, ParsisNLP]
#   3. Verb ending / stem normalization               [Assadi 2007, ParsisNLP, Miller 2011]
#   4. Copula reduction (است → ـه)                    [Assadi 2007, ParsisNLP]
#   5. Grammatical word reduction (را/یک/ها)           [Assadi 2007]
#   6. High-frequency lexical substitutions            [ParsisNLP, Miller 2011]
#
# (Numbering follows the analysis order from the literature review; the
# execution order below is Category 7 → 2 → 1 → 3 → 4 → 5 → 6, since each
# category's rules assume the previous categories have already normalized
# their inputs.)
# ============================================================================

import re

ZWNJ = '\u200c'   # Persian "half-space" (zero-width non-joiner)


def _wb(pattern: str) -> str:
    """
    Approximate Persian word boundary wrapper.

    Python's re module does not treat Arabic-script characters as \\w,
    so \\b does not work reliably for Persian. We instead use negative
    look-around on Persian letters and ZWNJ to approximate a word edge.
    """
    return r'(?<![ا-ی' + ZWNJ + r'])' + pattern + r'(?![ا-ی' + ZWNJ + r'])'


def normalize_to_colloquial(text: str) -> str:
    """
    Normalize a Persian transcript to a consistent colloquial orthography.

    Applied to BOTH reference transcripts AND model hypotheses before
    any WER/CER calculation, ensuring orthographic convention does not
    affect the score (only acoustic accuracy does).

    See module docstring for category sources and execution order.
    """

    if not isinstance(text, str):
        return text

    # ========================================================================
    # CATEGORY 7 — Arabic → Persian character normalization
    # Source: [Rasooli 2013] Sec 3.1 — standard Persian NLP tools expect
    # Persian Unicode; Arabic letters leak in via keyboard layout / OCR.
    # Must run FIRST, since every subsequent rule is written in Persian
    # Unicode and would silently fail to match Arabic variants.
    # Risk: Low.
    # ========================================================================
    text = text.replace('\u0643', '\u06A9')   # Arabic ك -> Persian ک
    text = text.replace('\u064A', '\u06CC')   # Arabic ي -> Persian ی
    text = text.replace('\u0629', '\u0647')   # Arabic ة -> Persian ه

    # ========================================================================
    # CATEGORY 2 — می / نمی ZWNJ spacing canonicalization
    # Source: [Rasooli 2013] Sec 3.1, Sec 4 — ~8% of tokens in the Persian
    # dependency treebank carry semi-spaces; real-world text frequently
    # substitutes a full space or no separator at all. [ParsisNLP] Sec 1.1.4
    # also notes می-prefixed verbs as a major source of formal/colloquial
    # divergence.
    # All three surface forms (ZWNJ / full space / none) are canonicalized
    # to "X‌" + ZWNJ before any verb-stem rule runs, so Category 3 rules
    # only need to handle one spacing convention.
    # Risk: Low.
    # ========================================================================

    # نمی (longer prefix first, to avoid a partial match against می)
    text = re.sub(r'نمی ([^\s' + ZWNJ + r'])', r'نمی' + ZWNJ + r'\1', text)
    text = re.sub(r'نمی([ا-ی])',                r'نمی' + ZWNJ + r'\1', text)

    # می
    text = re.sub(r'می ([^\s' + ZWNJ + r'])', r'می' + ZWNJ + r'\1', text)
    text = re.sub(r'می([ا-ی])',                r'می' + ZWNJ + r'\1', text)

    # ========================================================================
    # CATEGORY 1 — /ān/ → [un] vowel raising
    # Source: [Miller 2011] — the most extensively documented feature of
    # informal spoken Tehrani Persian; [ParsisNLP] Sec 1.1.1 gives the same
    # pattern for SDS purposes (xabarxan -> xabarxun).
    #
    # Risk: Medium. [Miller 2011] Sec 4.1 documents that raising is BLOCKED
    # by a morpheme boundary before the nasal (جانماز) and does not occur
    # before the velar nasal /ng/ (بانگ, دانگ). Some high-frequency words
    # also resist raising even in casual speech (دانشگاه, قرآن).
    # Per the literature's own caution, this rule is applied only to a
    # VERIFIED LIST of common closed-class words and noun/adjective stems,
    # not as a blind substitution over all -ان sequences.
    # ========================================================================
    aan_to_un_map = {
        'آن':        'اون',
        'آنها':      'اونا',
        'آنجا':      'اونجا',
        'همان':      'همون',
        'همانجا':    'همونجا',
        'چنان':      'چنون',
        'ایشان':     'اشون',
        'خودشان':    'خودشون',
        'خودمان':    'خودمون',
        'خودتان':    'خودتون',
        'مسلمان':    'مسلمون',
        'مهربان':    'مهربون',
        'خانه':      'خونه',
        'خانه‌ها':   'خونه‌ها',
        'جان':       'جون',
        'نان':       'نون',
        'تهران':     'تهرون',
        'ایران':     'ایرون',
    }
    for formal, colloquial in aan_to_un_map.items():
        text = re.sub(_wb(formal), colloquial, text)

    # ========================================================================
    # CATEGORY 3 — Verb ending / stem normalization
    # Source: [Assadi 2007] (3sg -d deletion at syllable end, the
    # statistical basis for ـَد -> ـه); [ParsisNLP] Sec 1.1.4 (می‌خورم
    # collapsing future/present forms); [Miller 2011] (a->u stem raising
    # combined with verb suffixation, e.g. /amadam/ -> [umaedaem]).
    #
    # 3a: 3sg present ـَد -> ـه  (formal می‌کند -> colloquial می‌کنه)
    # 3b: stem vowel a->u combined with suffixation (دان->دون, توان->تون)
    # 3c: 1sg/2sg contracted forms (می‌گویم->می‌گم, می‌خواهم->می‌خوام)
    #
    # These three sub-patterns are applied together per verb, since a
    # given verb typically needs both stem and ending normalization.
    # Risk: Low — each entry is a known, closed-set verb form.
    # ========================================================================
    verb_forms = {
        # کردن (to do)
        'می' + ZWNJ + 'کند':   'می' + ZWNJ + 'کنه',
        'نمی' + ZWNJ + 'کند':  'نمی' + ZWNJ + 'کنه',
        # شدن (to become)
        'می' + ZWNJ + 'شود':   'می' + ZWNJ + 'شه',
        'نمی' + ZWNJ + 'شود':  'نمی' + ZWNJ + 'شه',
        # رفتن (to go)
        'می' + ZWNJ + 'رود':   'می' + ZWNJ + 'ره',
        'نمی' + ZWNJ + 'رود':  'نمی' + ZWNJ + 'ره',
        # آمدن (to come)
        'می' + ZWNJ + 'آید':   'می' + ZWNJ + 'آد',
        'نمی' + ZWNJ + 'آید':  'نمی' + ZWNJ + 'آد',
        'آمدم':                'اومدم',
        'آمدی':                'اومدی',
        'آمد':                 'اومد',
        # گفتن (to say) — 1sg/2sg/3sg contraction [Cat 3c]
        'می' + ZWNJ + 'گویم':  'می' + ZWNJ + 'گم',
        'می' + ZWNJ + 'گویی':  'می' + ZWNJ + 'گی',
        'می' + ZWNJ + 'گوید':  'می' + ZWNJ + 'گه',
        'نمی' + ZWNJ + 'گوید': 'نمی' + ZWNJ + 'گه',
        # خواستن (to want) — 1sg/2sg/3sg contraction [Cat 3c]
        'می' + ZWNJ + 'خواهم': 'می' + ZWNJ + 'خوام',
        'می' + ZWNJ + 'خواهی': 'می' + ZWNJ + 'خوای',
        'می' + ZWNJ + 'خواهد': 'می' + ZWNJ + 'خواد',
        'نمی' + ZWNJ + 'خواهم':'نمی' + ZWNJ + 'خوام',
        'نمی' + ZWNJ + 'خواهد':'نمی' + ZWNJ + 'خواد',
        # دانستن (to know) — stem a->u [Cat 3b] + ending [Cat 3a]
        'می' + ZWNJ + 'دانم':  'می' + ZWNJ + 'دونم',
        'می' + ZWNJ + 'دانی':  'می' + ZWNJ + 'دونی',
        'می' + ZWNJ + 'داند':  'می' + ZWNJ + 'دونه',
        'نمی' + ZWNJ + 'دانم': 'نمی' + ZWNJ + 'دونم',
        'نمی' + ZWNJ + 'دانی': 'نمی' + ZWNJ + 'دونی',
        'نمی' + ZWNJ + 'داند': 'نمی' + ZWNJ + 'دونه',
        # توانستن (to be able) — stem a->u [Cat 3b] + ending [Cat 3a]
        'می' + ZWNJ + 'توانم': 'می' + ZWNJ + 'تونم',
        'می' + ZWNJ + 'توانی': 'می' + ZWNJ + 'تونی',
        'می' + ZWNJ + 'تواند': 'می' + ZWNJ + 'تونه',
        'نمی' + ZWNJ + 'توانم':'نمی' + ZWNJ + 'تونم',
        'نمی' + ZWNJ + 'تواند':'نمی' + ZWNJ + 'تونه',
        # دیدن (to see)
        'می' + ZWNJ + 'بیند':  'می' + ZWNJ + 'بینه',
        'نمی' + ZWNJ + 'بیند': 'نمی' + ZWNJ + 'بینه',
        # خوردن (to eat/drink)
        'می' + ZWNJ + 'خورد':  'می' + ZWNJ + 'خوره',
        'نمی' + ZWNJ + 'خورد': 'نمی' + ZWNJ + 'خوره',
        # زدن (to hit/strike)
        'می' + ZWNJ + 'زند':   'می' + ZWNJ + 'زنه',
        'نمی' + ZWNJ + 'زند':  'نمی' + ZWNJ + 'زنه',
    }
    for formal, colloquial in verb_forms.items():
        text = re.sub(_wb(formal), colloquial, text)

    # ========================================================================
    # CATEGORY 4 — Copula reduction (است -> ـه)
    # Source: [Assadi 2007] Sec 3 — explicitly documents formal "gol ast" ->
    # colloquial "gol-e", and attributes the high frequency of [e] in
    # colloquial Persian partly to this change. [ParsisNLP] confirms the
    # general formal-vs-colloquial register split.
    #
    # Risk: Medium. The fused adjective+است written form requires knowing
    # the word boundary; per Category 4's documented risk, this is
    # restricted to a verified list of common adjectives rather than a
    # blind "X است -> Xه" rule across all words.
    # ========================================================================
    copula_map = {
        'خوب است':    'خوبه',
        'سخت است':    'سخته',
        'راحت است':   'راحته',
        'درست است':   'درسته',
        'غلط است':    'غلطه',
        'بد است':     'بده',
        'قشنگ است':   'قشنگه',
        'چطور است':   'چطوره',
        'خوبه':       'خوبه',     # already colloquial — identity (idempotency)
        'سخته':       'سخته',
        'راحته':      'راحته',
        'درسته':      'درسته',
        'غلطه':       'غلطه',
        'بده':        'بده',
        'قشنگه':      'قشنگه',
        'چطوره':      'چطوره',
    }
    for formal, colloquial in copula_map.items():
        text = re.sub(_wb(formal), colloquial, text)

    # Bare existential/negative copula forms — same in formal and
    # colloquial speech, kept here as explicit identity rules so the
    # normalizer's behavior on these very common tokens is documented
    # and stable (idempotency).
    for form in ['هستم', 'هستی', 'هست', 'نیستم', 'نیستی', 'نیست']:
        text = re.sub(_wb(form), form, text)

    # ========================================================================
    # CATEGORY 5 — Grammatical word reduction
    # Source: [Assadi 2007] Sec 5 — 74% of all phoneme deletions in colloquial
    # Persian occur in grammatical words. Specific documented cases:
    #   - را -> رو   (37/38 of /r/-deletions occur in this particle)
    #   - یک -> یه   (30/31 of /k/-deletions occur in this article)
    #   - ـها -> ـا  (37/41 of /h/-deletions occur in this plural suffix)
    # Risk: یک/را Low (closed-class, high-frequency). ـها->ـا Medium
    # (broad suffix substitution risks over-application), so it is
    # restricted to the explicit plural marker token only.
    # ========================================================================
    text = re.sub(_wb('را'), 'رو', text)
    text = re.sub(_wb('یک'), 'یه', text)
    # Plural ها -> ا, only as a separated/ZWNJ-attached suffix token
    # (lookbehind requires a preceding Persian letter; lookahead blocks a
    # following Persian letter/ZWNJ so we don't match inside a longer word)
    text = re.sub(
        r'(?<=[ا-ی])' + ZWNJ + r'ها(?![ا-ی' + ZWNJ + r'])', ZWNJ + 'ا', text
    )
    text = re.sub(
        r'(?<=[ا-ی]) ها(?![ا-ی' + ZWNJ + r'])', ZWNJ + 'ا', text
    )

    # ========================================================================
    # CATEGORY 6 — High-frequency lexical substitutions
    # Source: [ParsisNLP] Sec 1.1.2 (preposition removal), Sec 1.1.3
    # (formal/informal word pairs); [Miller 2011] (اونجا, خونه via /an/
    # raising, cross-referenced with Category 1); [Assadi 2007] (برای ->
    # برا via vowel/consonant deletion).
    # Risk: Low — closed list of common words.
    # ========================================================================
    lexical_map = {
        'اگر':    'اگه',
        'دیگر':   'دیگه',
        'اکنون':  'الان',
        'برای':   'برا',
        'هرگز':   'هیچوقت',
        'بسیار':  'خیلی',
        'چیزی':   'چیزی',     # unchanged — kept for documentation/idempotency
        'کجا':    'کجا',      # unchanged
    }
    for formal, colloquial in lexical_map.items():
        text = re.sub(_wb(formal), colloquial, text)

    # ------------------------------------------------------------------
    # Final cleanup: remove stray spaces left around ZWNJ, collapse
    # repeated spaces, strip leading/trailing whitespace.
    # ------------------------------------------------------------------
    text = re.sub(r' ' + ZWNJ, ZWNJ, text)
    text = re.sub(ZWNJ + r' ', ZWNJ, text)
    text = re.sub(r' +', ' ', text)
    text = text.strip()

    return text


# ---- Normalizer self-test: one example per category ------------------------
test_cases = [
    # (input, category being tested)
    ('\u0643\u062A\u0627\u0628',        'Cat 7: Arabic kaf -> Persian kaf'),
    ('می کنه',                          'Cat 2: full-space می -> ZWNJ'),
    ('میکنه',                           'Cat 2: no-sep می -> ZWNJ'),
    ('آن خانه',                         'Cat 1: آن->اون, خانه->خونه'),
    ('می‌کند',                          'Cat 3a: formal 3sg -> colloquial'),
    ('می‌دانم',                         'Cat 3b: stem a->u + ending'),
    ('می‌خواهم',                        'Cat 3c: 1sg contraction'),
    ('خوب است',                         'Cat 4: copula reduction'),
    ('کتاب را یک بار خواندم',           'Cat 5: را->رو, یک->یه'),
    ('کتاب‌ها',                         'Cat 5: ـها->ـا plural'),
    ('اگر دیگر برای او بیاید',          'Cat 6: lexical substitutions'),
    ('نمی‌دانم که می‌کند',              'Multi-category sentence'),
]

print("=" * 70)
print("Normalizer self-test (one example per category)")
print("=" * 70)
for text, label in test_cases:
    result = normalize_to_colloquial(text)
    print(f"[{label}]")
    print(f"  In : {text}")
    print(f"  Out: {result}")
print()

print("Idempotency check (normalize(normalize(x)) == normalize(x)):")
all_pass = True
for text, _ in test_cases:
    once  = normalize_to_colloquial(text)
    twice = normalize_to_colloquial(once)
    ok = (once == twice)
    all_pass &= ok
    print(f"  {'OK' if ok else 'FAILED'}  {text[:30]}")
print(f"\nAll idempotency checks passed: {all_pass}")

Normalizer self-test (one example per category)
[Cat 7: Arabic kaf -> Persian kaf]
  In : كتاب
  Out: کتاب
[Cat 2: full-space می -> ZWNJ]
  In : می کنه
  Out: می‌کنه
[Cat 2: no-sep می -> ZWNJ]
  In : میکنه
  Out: می‌کنه
[Cat 1: آن->اون, خانه->خونه]
  In : آن خانه
  Out: اون خونه
[Cat 3a: formal 3sg -> colloquial]
  In : می‌کند
  Out: می‌کنه
[Cat 3b: stem a->u + ending]
  In : می‌دانم
  Out: می‌دونم
[Cat 3c: 1sg contraction]
  In : می‌خواهم
  Out: می‌خوام
[Cat 4: copula reduction]
  In : خوب است
  Out: خوبه
[Cat 5: را->رو, یک->یه]
  In : کتاب را یک بار خواندم
  Out: کتاب رو یه بار خواندم
[Cat 5: ـها->ـا plural]
  In : کتاب‌ها
  Out: کتاب‌ا
[Cat 6: lexical substitutions]
  In : اگر دیگر برای او بیاید
  Out: اگه دیگه برا او بیاید
[Multi-category sentence]
  In : نمی‌دانم که می‌کند
  Out: نمی‌دونم که می‌کنه

Idempotency check (normalize(normalize(x)) == normalize(x)):
  OK  كتاب
  OK  می کنه
  OK  میکنه
  OK  آن خانه
  OK  می‌کند
  OK  می‌دانم
  OK  می‌خواهم
  OK  خوب است
  OK  کتاب را یک 

In [ ]:
# STAGE 4: APPLY NORMALIZATION TO FULL DATASET
# ============================================================================

def add_colloquial_column(sample):
    """Add text_colloquial column to each sample."""
    sample["text_colloquial"] = normalize_to_colloquial(sample["text"])
    return sample

dataset_full = dataset_full.map(add_colloquial_column)

# Count how many samples were actually changed
changed = sum(
    1 for i in range(len(dataset_full))
    if dataset_full[i]["text"] != dataset_full[i]["text_colloquial"]
)
print(f"\nSamples changed by normalization: {changed} / {len(dataset_full)} "
      f"({100*changed/len(dataset_full):.1f}%)")

# Inspect a sample of changed examples
print("\nSample of changed transcripts:")
shown = 0
for i in range(len(dataset_full)):
    orig = dataset_full[i]["text"]
    coll = dataset_full[i]["text_colloquial"]
    if orig != coll:
        print(f"  Sample {i}")
        print(f"    Original  : {orig}")
        print(f"    Colloquial: {coll}")
        shown += 1
        if shown >= 10:
            break

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]


Samples changed by normalization: 1464 / 3000 (48.8%)

Sample of changed transcripts:
  Sample 1
    Original  : برای اولین بار در دانشکده او را دیدم.
    Colloquial: برا اولین بار در دانشکده او رو دیدم.
  Sample 2
    Original  : با تنهایی که از درون او را آزار میداد،
    Colloquial: با تنهایی که از درون او رو آزار می‌داد،
  Sample 3
    Original  : آنها به شما حق نمی دهند مخالف باشید ، و نه بگویید.
    Colloquial: اونا به شما حق نمی‌دهند مخالف باشید ، و نه بگویید.
  Sample 4
    Original  : و یا حتی نخواهد این کار را انجام دهد.
    Colloquial: و یا حتی نخواهد این کار رو انجام دهد.
  Sample 5
    Original  : ولی آنها این حق را به خود میدهند
    Colloquial: ولی اونا این حق رو به خود می‌دهند
  Sample 6
    Original  : آنها هرگز دوست شما نیستند.
    Colloquial: اونا هیچوقت دوست شما نیستند.
  Sample 7
    Original  : ساعت پنج می بینمت
    Colloquial: ساعت پنج می‌بینمت
  Sample 11
    Original  : گاهی از شما میخواهند برای آنها یک برنامه کامپیوتری بنویسید
    Colloquial: گاهی از شما می‌خوا

In [ ]:
# STAGE 5: FIXED TRAIN / TEST SPLIT
#
# We split ONCE and reuse the same split for all four conditions so
# results are comparable. The test set (1,000 samples) is held out
# from both fine-tuning runs.
# ============================================================================

split = dataset_full.train_test_split(test_size=0.1, seed=42)
# train_dataset: 9,000 samples — used for fine-tuning
# test_dataset:  1,000 samples — used for all evaluations
train_dataset = split["train"]
test_dataset  = split["test"]

print(f"Train samples : {len(train_dataset)}")
print(f"Test  samples : {len(test_dataset)}")

Train samples : 2700
Test  samples : 300


In [ ]:
# STAGE 6: LOAD BASELINE MODEL
# ============================================================================

import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small", language="fa", task="transcribe"
)
baseline_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-small"
).to(device)
baseline_model.generation_config.language = "fa"
baseline_model.generation_config.task = "transcribe"
baseline_model.generation_config.forced_decoder_ids = None

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

print("Baseline Whisper Small loaded.")


def transcribe_batch(samples, model, proc, batch_size=16):
    """
    Transcribe a list of dataset samples in batches.
    Returns a list of hypothesis strings.
    """
    hypotheses = []
    for start in range(0, len(samples), batch_size):
        batch = samples[start : start + batch_size]
        arrays  = [s["audio"]["array"] for s in batch]
        rates   = [s["audio"]["sampling_rate"] for s in batch]

        inputs = proc(
            arrays,
            sampling_rate=rates[0],
            return_tensors="pt",
            padding=True
        ).input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(inputs)

        decoded = proc.batch_decode(predicted_ids, skip_special_tokens=True)
        hypotheses.extend(decoded)

        if (start + batch_size) % 100 == 0 or (start + batch_size) >= len(samples):
            print(f"  Transcribed {min(start+batch_size, len(samples))}/{len(samples)}")

    return hypotheses


def compute_wer_cer(hypotheses, references):
    """
    Compute WER and CER after applying normalize_to_colloquial() to
    BOTH hypotheses and references.

    This is the single evaluation function used for ALL four conditions.
    Normalizing both sides ensures we measure acoustic accuracy and not
    orthographic convention differences (spacing, ZWNJ, Arabic chars, etc.).
    """
    norm_hyps = [normalize_to_colloquial(h) for h in hypotheses]
    norm_refs = [normalize_to_colloquial(r) for r in references]
    wer = wer_metric.compute(predictions=norm_hyps, references=norm_refs)
    cer = cer_metric.compute(predictions=norm_hyps, references=norm_refs)
    return wer, cer

Using device: cpu


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Baseline Whisper Small loaded.


In [ ]:
# STAGE 7: CONDITION 1 — BASELINE
# (no fine-tuning; normalize both sides before scoring)
# ============================================================================

print("\n--- Condition 1: Baseline ---")
test_samples  = [test_dataset[i] for i in range(len(test_dataset))]
test_refs_orig = [s["text"] for s in test_samples]

baseline_hyps = transcribe_batch(test_samples, baseline_model, processor)

# Normalize BOTH sides before WER/CER (supervisor's feedback)
cond1_wer, cond1_cer = compute_wer_cer(baseline_hyps, test_refs_orig)
print(f"Condition 1  WER: {cond1_wer:.4f}  CER: {cond1_cer:.4f}")


--- Condition 1: Baseline ---


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transform

KeyboardInterrupt: 

In [ ]:
# STAGE 8: CONDITION 2 — COLLOQUIAL REFERENCES ONLY (no fine-tuning)
#
# Same baseline model, same hypotheses as Condition 1.
# Only difference: references are the colloquial-normalized versions.
# Both sides are still normalized before scoring (as always).
#
# Purpose: isolate the effect of reference normalization alone,
# without any model retraining.
# ============================================================================

print("\n--- Condition 2: Colloquial references only (no fine-tuning) ---")
test_refs_coll = [s["text_colloquial"] for s in test_samples]

# Hypotheses are the same as Condition 1 — no new transcription needed.
# compute_wer_cer normalizes both sides, so we pass colloquial refs here.
cond2_wer, cond2_cer = compute_wer_cer(baseline_hyps, test_refs_coll)
print(f"Condition 2  WER: {cond2_wer:.4f}  CER: {cond2_cer:.4f}")


--- Condition 2: Colloquial references only (no fine-tuning) ---
Condition 2  WER: 0.9446  CER: 0.4195


In [ ]:
# STAGE 9: FINE-TUNING SETUP
#
# DataCollatorSpeechSeq2SeqWithPadding and compute_metrics are adapted
# from Hameed et al. (2025) — DOLMA-NLP/asr — finetune_whisper.py.
# ============================================================================

import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """
    Pads input features and labels to the same length within each batch.
    Padding tokens in labels are replaced with -100 so they are ignored
    by the cross-entropy loss.

    Adapted from Hameed et al. (2025), DOLMA-NLP/asr, finetune_whisper.py.
    """
    processor: Any
    decoder_start_token_id: int

    def __call__(
        self, features: List[Dict[str, Union[List[int], torch.Tensor]]]
    ) -> Dict[str, torch.Tensor]:

        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(
            input_features, return_tensors="pt"
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features, return_tensors="pt"
        )

        # Replace padding with -100 so it is ignored in loss computation
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Remove BOS token if the tokenizer prepended it
        # (Whisper prepends it; it is added again during generation)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


def make_compute_metrics(proc):
    """
    Returns a compute_metrics function for Seq2SeqTrainer.

    Both predicted strings and label strings are passed through
    normalize_to_colloquial() before WER/CER computation, consistent
    with the evaluation rule used in all four conditions.

    Structure adapted from Hameed et al. (2025), DOLMA-NLP/asr,
    finetune_whisper.py — compute_metrics function.
    """
    def compute_metrics(pred):
        pred_ids  = pred.predictions
        label_ids = pred.label_ids

        # Replace -100 (masked padding) with the pad token id
        label_ids[label_ids == -100] = proc.tokenizer.pad_token_id

        pred_str  = proc.tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
        label_str = proc.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        # Normalize BOTH sides before computing metrics
        pred_str_norm  = [normalize_to_colloquial(p) for p in pred_str]
        label_str_norm = [normalize_to_colloquial(l) for l in label_str]

        wer = wer_metric.compute(predictions=pred_str_norm, references=label_str_norm)
        cer = cer_metric.compute(predictions=pred_str_norm, references=label_str_norm)
        return {"wer": wer, "cer": cer}

    return compute_metrics


def prepare_features(sample, proc, text_column):
    """Extract audio features and tokenize transcript for one sample."""
    audio = sample["audio"]
    inputs = proc.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    )
    sample["input_features"] = inputs.input_features[0]
    sample["labels"] = proc.tokenizer(sample[text_column]).input_ids
    return sample


def run_finetuning(
    train_ds,
    eval_ds,
    text_column,
    output_dir,
    num_steps=500
):
    """
    Fine-tune Whisper Small on train_ds using transcripts from text_column.
    Returns the fine-tuned model and its processor.

    Training arguments follow Hameed et al. (2025), DOLMA-NLP/asr,
    train.sh and finetune_whisper.py configuration.
    """
    proc = WhisperProcessor.from_pretrained(
        "openai/whisper-small", language="fa", task="transcribe"
    )
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
    model.generation_config.language = "fa"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None

    # Prepare datasets
    cols_to_remove = train_ds.column_names
    train_prepared = train_ds.map(
        lambda s: prepare_features(s, proc, text_column),
        remove_columns=cols_to_remove
    )
    eval_prepared = eval_ds.map(
        lambda s: prepare_features(s, proc, text_column),
        remove_columns=eval_ds.column_names
    )

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=proc,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

    # Training arguments adapted from Hameed et al. (2025)
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=2,      # effective batch size = 16
        learning_rate=1e-5,
        warmup_steps=50,
        max_steps=num_steps,
        gradient_checkpointing=True,
        fp16=True,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=225,
        report_to=["tensorboard"],
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_prepared,
        eval_dataset=eval_prepared,
        data_collator=data_collator,
        compute_metrics=make_compute_metrics(proc),
        processing_class=proc.feature_extractor,  # as per Hameed et al. 2025
    )

    print(f"  Starting fine-tuning -> {output_dir} (text column: '{text_column}')")
    trainer.train()
    trainer.save_model(output_dir)
    proc.save_pretrained(output_dir)
    print(f"  Fine-tuning complete. Model saved to {output_dir}")

    return trainer.model, proc

In [ ]:
# STAGE 10: CONDITION 3 — FINE-TUNED ON ORIGINAL TRANSCRIPTS
# ============================================================================

print("\n--- Condition 3: Fine-tune on original (formal/noisy) transcripts ---")

ft_original_model, ft_original_proc = run_finetuning(
    train_ds=train_dataset,
    eval_ds=test_dataset,
    text_column="text",                       # original transcripts
    output_dir="./whisper-small-fa-original",
)

print("Transcribing test set with Condition 3 model...")
cond3_hyps = transcribe_batch(test_samples, ft_original_model, ft_original_proc)

# Normalize BOTH sides before scoring
cond3_wer, cond3_cer = compute_wer_cer(cond3_hyps, test_refs_orig)
print(f"Condition 3  WER: {cond3_wer:.4f}  CER: {cond3_cer:.4f}")


--- Condition 3: Fine-tune on original (formal/noisy) transcripts ---


NameError: name 'run_finetuning' is not defined

In [ ]:
# STAGE 11: CONDITION 4 — FINE-TUNED ON COLLOQUIAL TRANSCRIPTS
# ============================================================================

print("\n--- Condition 4: Fine-tune on colloquial-normalized transcripts ---")

ft_colloquial_model, ft_colloquial_proc = run_finetuning(
    train_ds=train_dataset,
    eval_ds=test_dataset,
    text_column="text_colloquial",            # colloquial-normalized transcripts
    output_dir="./whisper-small-fa-colloquial",
)

print("Transcribing test set with Condition 4 model...")
cond4_hyps = transcribe_batch(test_samples, ft_colloquial_model, ft_colloquial_proc)

# Normalize BOTH sides before scoring
cond4_wer, cond4_cer = compute_wer_cer(cond4_hyps, test_refs_coll)
print(f"Condition 4  WER: {cond4_wer:.4f}  CER: {cond4_cer:.4f}")

In [ ]:
# STAGE 12: FOUR-WAY COMPARISON
# ============================================================================

def pct(delta, base):
    """Percentage relative reduction."""
    return (delta / base) * 100 if base else float('nan')


print("\n" + "=" * 72)
print(f"{'Condition':<42} {'WER':>8} {'CER':>8}")
print("=" * 72)
print(f"{'1. Baseline (no FT, original refs)':<42} {cond1_wer:>8.4f} {cond1_cer:>8.4f}")
print(f"{'2. Colloquial refs only (no FT)':<42} {cond2_wer:>8.4f} {cond2_cer:>8.4f}")
print(f"{'3. Fine-tuned on original transcripts':<42} {cond3_wer:>8.4f} {cond3_cer:>8.4f}")
print(f"{'4. Fine-tuned on colloquial transcripts':<42} {cond4_wer:>8.4f} {cond4_cer:>8.4f}")
print("=" * 72)

# --- Pairwise comparisons ---
print("\nPairwise comparisons (absolute delta and relative % reduction):\n")

# Effect of fine-tuning alone (on original transcripts): Cond 1 -> Cond 3
d_wer_13 = cond1_wer - cond3_wer
d_cer_13 = cond1_cer - cond3_cer
print(f"Effect of fine-tuning (original transcripts) [1 -> 3]:")
print(f"  WER: {d_wer_13:+.4f}  ({pct(d_wer_13, cond1_wer):+.1f}% relative)")
print(f"  CER: {d_cer_13:+.4f}  ({pct(d_cer_13, cond1_cer):+.1f}% relative)")

# Effect of fine-tuning + colloquial normalization: Cond 1 -> Cond 4
d_wer_14 = cond1_wer - cond4_wer
d_cer_14 = cond1_cer - cond4_cer
print(f"\nEffect of fine-tuning + colloquial normalization [1 -> 4]:")
print(f"  WER: {d_wer_14:+.4f}  ({pct(d_wer_14, cond1_wer):+.1f}% relative)")
print(f"  CER: {d_cer_14:+.4f}  ({pct(d_cer_14, cond1_cer):+.1f}% relative)")

# Isolated effect of colloquial normalization (controlling for FT): Cond 3 -> Cond 4
d_wer_34 = cond3_wer - cond4_wer
d_cer_34 = cond3_cer - cond4_cer
print(f"\nIsolated effect of colloquial normalization [3 -> 4]:")
print(f"  WER: {d_wer_34:+.4f}  ({pct(d_wer_34, cond3_wer):+.1f}% relative)")
print(f"  CER: {d_cer_34:+.4f}  ({pct(d_cer_34, cond3_cer):+.1f}% relative)")

# Effect of reference normalization alone (no FT): Cond 1 -> Cond 2
d_wer_12 = cond1_wer - cond2_wer
d_cer_12 = cond1_cer - cond2_cer
print(f"\nEffect of reference normalization alone (no FT) [1 -> 2]:")
print(f"  WER: {d_wer_12:+.4f}  ({pct(d_wer_12, cond1_wer):+.1f}% relative)")
print(f"  CER: {d_cer_12:+.4f}  ({pct(d_cer_12, cond1_cer):+.1f}% relative)")

print("\n[All WER/CER values computed after normalize_to_colloquial() applied")
print(" to BOTH hypothesis and reference — consistent across all conditions.]")

# ============================================================================
# PROJECT COMPLETE